In [46]:
import cv2
import ffmpeg
import ipywidgets as widgets
import json
import mediapipe as mp
import numpy as np
import pyttsx3
import re
import sounddevice as sd
import threading
import time
import vosk
from typing import Any, Dict, cast

from datetime import datetime
from IPython.display import display
from pathlib import Path
from collections import deque
import subprocess

import csv
import collections
import queue
import edge_tts
import asyncio
import tempfile
import pygame
import psutil, os




In [51]:

MODEL_PATH = "pose_landmarker_full.task"
VOSK_MODEL_PATH = "vosk-model-small-pl-0.22"

FRONT_STREAM_URL = "front.mp4"
SIDE_STREAM_URL = "side.mp4"

STATE_CONFIRM_FRAMES = 3
MIN_REP_INTERVAL = 1.0

ANGLE_WINDOW = 5
angle_buffer = deque(maxlen=ANGLE_WINDOW)

MAX_WIDTH_PER_CAMERA = 640
JPEG_QUALITY = 65
FPS = 40
TIMEOUT = 0

HYSTERESIS = 8

vosk.SetLogLevel(-1)

speech_lock = threading.Lock()
AUTO_SAVE_TO_FILE = True
AUTO_START_SERIES = False
BOTH_CAMERAS_REQUIRED = False

YOLO_MODEL_PATH = "yolo11n.pt"
YOLO_CONF = 1.1
_yolo_model = None

def get_yolo():
	global _yolo_model
	if _yolo_model is None:
		from ultralytics import YOLO
		_yolo_model = YOLO(YOLO_MODEL_PATH)
	return _yolo_model

_yolo_results = {"front": None, "side": None}
_yolo_lock = threading.Lock()

def yolo_worker():
	last_frames = {"front": None, "side": None}
	while not stop.is_set():
		for cam in ["front", "side"]:
			with state_lock:
				frame = latest_frames[cam]
			if frame is None or frame is last_frames[cam]:
				continue
			last_frames[cam] = frame
			f = cv2.resize(frame, (320, 240))
			_, bbox = crop_person(f)
			if bbox is not None:
				sx = frame.shape[1] / 320
				sy = frame.shape[0] / 240
				x1, y1, x2, y2 = bbox
				bbox = (int(x1*sx), int(y1*sy), int(x2*sx), int(y2*sy))
			with _yolo_lock:
				_yolo_results[cam] = bbox
		time.sleep(0.15)

def crop_person(frame):
	model = get_yolo()
	results = model(frame, classes=[0], conf=YOLO_CONF, verbose=False)
	boxes = results[0].boxes
	if boxes is None or len(boxes) == 0:
		return frame, None
	b = boxes.xyxy[0].cpu().numpy().astype(int)
	x1, y1, x2, y2 = b
	h, w = frame.shape[:2]
	x1, y1 = max(0, x1), max(0, y1)
	x2, y2 = min(w, x2), min(h, y2)
	return frame, (x1, y1, x2, y2)

GT_MODE = True
gt_manual_reps = 0
gt_lock = threading.Lock()

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

POSE_CONNECTIONS = [
	(0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8),
	(9, 10), (11, 12), (11, 23), (12, 24), (23, 24),
	(11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19),
	(12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
	(23, 25), (25, 27), (27, 29), (29, 31),
	(24, 26), (26, 28), (28, 30), (30, 32),
	(27, 31), (28, 32),
]

EXERCISE_DEFINITIONS: Dict[str, Dict[str, Any]] = {
	"dumbbell_curl": {
		"label": "Podnoszenie hantli",
		"rep_rule": "Pelny ruch: lokiec < 40 stopni i > 140 stopni",
		"down_threshold": 60,
		"up_threshold": 120,
		"min_amplitude": 60,
		"symmetry_threshold": 25,
		"sway_threshold": 25,
		"elbow_threshold": 35,
		"notes": [
			"Kamera frontowa sprawdza symetrie ramion.",
			"Kamera boczna stabilizuje liczenie faz ruchu.",
			"Komendy: start / seria / reset / stop / zapis on / zapis off.",
			"W sessions/ zapisywany jest film i JSON dla kazdej serii.",
		],
	}
}
ACTIVE_EXERCISE_KEY = "dumbbell_curl"
VOICE_COMMANDS = ["start", "seria", "reset", "stop", "zapis on", "zapis off"]

START_WORDS = ["start", "zaczynam", "zacznij", "zacznijmy", "startujemy"]
STOP_WORDS = ["stop", "koniec", "zatrzymaj", "skończ", "skoncz", "koncz"]
SERIES_WORDS = ["seria", "serie", "serię", "serii"]

vosk_model = vosk.Model(VOSK_MODEL_PATH)

stop = threading.Event()
state_lock = threading.Lock()


tts_queue = queue.Queue()

latest_frames: Dict[str, Any] = {"front": None, "side": None}
latest_landmarks: Dict[str, Any] = {"front": None, "side": None}

latest_timestamps = {"front": 0, "side": 0}
processed_timestamps = {"front": 0, "side": 0}



fps_counters = {
	"front": collections.deque(maxlen=60),
	"side": collections.deque(maxlen=60),
	"process": collections.deque(maxlen=60),
}

_angle_log_lock = threading.Lock()
_angle_log_buffer: list = []
_angle_log_path: str = ""

_confidence_log_buffer: list = []

def get_ram_usage():
	proc = psutil.Process(os.getpid())
	return round(proc.memory_info().rss / 1024**2, 1)

_gpu_cache = (None, None)

def _gpu_worker():
	global _gpu_cache
	while not stop.is_set():
		try:
			result = subprocess.run(
				["radeontop", "-d", "-", "-l", "1"],
				capture_output=True, text=True, timeout=3
			)
			for line in result.stdout.splitlines():
				if "gpu" in line.lower():
					m = re.search(r"gpu\s+([\d.]+)%", line.lower())
					if m:
						_gpu_cache = (None, float(m.group(1)))
						break
		except Exception:
			pass
		time.sleep(2)

def get_gpu_usage():
	return _gpu_cache


def _init_logs():
	"""Tworzy pliki CSV na początku sesji."""
	global _angle_log_path
	Path("sessions").mkdir(exist_ok=True)
	ts = datetime.now().strftime("%Y%m%d_%H%M%S")
	_angle_log_path = f"sessions/angles_{ts}.csv"
	with open(_angle_log_path, "w", newline="", encoding="utf-8") as f:
		writer = csv.writer(f)
		writer.writerow(["timestamp", "left_angle", "right_angle", "avg_angle", "smoothed_angle", "phase", "source"])


def _log_angle(left, right, avg, smoothed, phase, source):
	"""Buforuje wpis kąta — zapis do CSV co 100 klatek."""
	global _angle_log_buffer
	entry = {
		"timestamp": round(time.time(), 4),
		"left_angle": round(left, 2) if left is not None else "",
		"right_angle": round(right, 2) if right is not None else "",
		"avg_angle": round(avg, 2) if avg is not None else "",
		"smoothed_angle": round(smoothed, 2) if smoothed is not None else "",
		"phase": phase,
		"source": source,
	}
	with _angle_log_lock:
		_angle_log_buffer.append(entry)
		if len(_angle_log_buffer) >= 100:
			_flush_angle_log()


def _flush_angle_log():
	"""Zapisuje bufor do CSV. Wywołuj pod _angle_log_lock."""
	if not _angle_log_buffer or not _angle_log_path:
		return
	with open(_angle_log_path, "a", newline="", encoding="utf-8") as f:
		writer = csv.DictWriter(f, fieldnames=["timestamp", "left_angle", "right_angle", "avg_angle", "smoothed_angle", "phase", "source"])
		writer.writerows(_angle_log_buffer)
	_angle_log_buffer.clear()


def _log_confidence(camera, landmarks):
	"""Loguje visibility score dla kluczowych landmarków (11,12,13,14,15,16,23,24)."""
	key_indices = [11, 12, 13, 14, 15, 16, 23, 24]
	entry = {"timestamp": round(time.time(), 4), "camera": camera}
	for idx in key_indices:
		try:
			entry[f"lm{idx}_vis"] = round(landmarks[idx].visibility, 3)
		except Exception:
			entry[f"lm{idx}_vis"] = ""
	with _angle_log_lock:
		_confidence_log_buffer.append(entry)


def _flush_confidence_log(path_prefix):
	"""Zapisuje confidence log do osobnego CSV."""
	if not _confidence_log_buffer:
		return
	conf_path = f"{path_prefix}_confidence.csv"
	fieldnames = ["timestamp", "camera"] + [f"lm{i}_vis" for i in [11, 12, 13, 14, 15, 16, 23, 24]]
	with open(conf_path, "w", newline="", encoding="utf-8") as f:
		writer = csv.DictWriter(f, fieldnames=fieldnames)
		writer.writeheader()
		writer.writerows(_confidence_log_buffer)


def get_fps(camera_name):
	times = fps_counters[camera_name]
	if len(times) < 2:
		return 0.0
	return round((len(times) - 1) / (times[-1] - times[0]), 1) if times[-1] != times[0] else 0.0


dumbbell_state: Dict[str, Any] = {
	"phase": "extended",
	"reps": 0,
	"cycle_min": 180.0,
	"cycle_max": 0.0,
	"amplitude_warned": False,
	"symmetry_warned": False,
	"sway_warned": False,
	"elbow_warned": False,
	"candidate_phase": None,
	"candidate_count": 0,
	"last_rep_time": 0
}

recording: Dict[str, Any] = {
	"active": False,
	"frames": [],
	"video_writer": None,
	"video_path": None,
	"stats_by_rep": {},
	"series_index": 1,
	"started_at": None,
}

save_cfg: Dict[str, Any] = {"enabled": AUTO_SAVE_TO_FILE}

img_widget = widgets.Image(format="jpeg", layout=widgets.Layout(
	border="2px solid #1a1a2e",
	border_radius="12px",
	width="100%",
))
status_html = widgets.HTML(value="")
stats_html = widgets.HTML(value="")
voice_html = widgets.HTML(value="")
assumptions_html = widgets.HTML(value="")
metrics_html = widgets.HTML(value="")


def get_active_exercise():
	return EXERCISE_DEFINITIONS[ACTIVE_EXERCISE_KEY]

def _create_ffmpeg_writer(path, w, h, fps):
	cmd = [
		"ffmpeg", "-y",
		"-f", "rawvideo",
		"-vcodec", "rawvideo",
		"-pix_fmt", "bgr24",
		"-s", f"{w}x{h}",
		"-r", str(fps),
		"-i", "pipe:0",
		"-vcodec", "libx264",
		"-preset", "ultrafast",
		"-pix_fmt", "yuv420p",
		path
	]
	return subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=subprocess.DEVNULL)


def tts_worker():

	pygame.mixer.init()
	loop = asyncio.new_event_loop()
	asyncio.set_event_loop(loop)

	async def say(text):
		communicate = edge_tts.Communicate(text, "pl-PL-MarekNeural")
		with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as f:
			path = f.name
		await communicate.save(path)
		with speech_lock:
			pygame.mixer.music.load(path)
			pygame.mixer.music.play()
			while pygame.mixer.music.get_busy():
				pygame.time.wait(50)
		Path(path).unlink()

	while not stop.is_set():
		try:
			text = tts_queue.get(timeout=0.2)
			loop.run_until_complete(say(text))
			tts_queue.task_done()
		except queue.Empty:
			continue

	loop.close()

def speak(text):
	tts_queue.put(text)


def angle(a, b, c):
	pa = np.array([a.x, a.y])
	pb = np.array([b.x, b.y])
	pc = np.array([c.x, c.y])
	ba = pa - pb
	bc = pc - pb
	den = (np.linalg.norm(ba) * np.linalg.norm(bc))
	if den == 0:
		return 180.0
	cosv = np.dot(ba, bc) / den
	return float(np.degrees(np.arccos(np.clip(cosv, -1.0, 1.0))))


def arm_angles(landmarks):
	left = angle(landmarks[11], landmarks[13], landmarks[15])
	right = angle(landmarks[12], landmarks[14], landmarks[16])
	return left, right, (left + right) / 2.0


def draw_pose(frame, landmarks, line_color=(0, 255, 136), point_color=(255, 100, 100)):
	h, w = frame.shape[:2]
	for start_idx, end_idx in POSE_CONNECTIONS:
		p1 = landmarks[start_idx]
		p2 = landmarks[end_idx]
		cv2.line(frame, (int(p1.x * w), int(p1.y * h)), (int(p2.x * w), int(p2.y * h)), line_color, 2)
	for lm in landmarks:
		cv2.circle(frame, (int(lm.x * w), int(lm.y * h)), 4, point_color, -1)


def resize_frame(frame, max_width):
	h, w = frame.shape[:2]
	if w <= max_width:
		return frame
	return cv2.resize(frame, (max_width, int(h * max_width / w)))


def compose_frames(front_frame, side_frame):
	if front_frame is None and side_frame is None:
		return None

	if front_frame is None:
		front_frame = np.zeros((480, MAX_WIDTH_PER_CAMERA, 3), dtype=np.uint8)
		cv2.putText(front_frame, "BRAK FRONT", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
	if side_frame is None:
		side_frame = np.zeros((480, MAX_WIDTH_PER_CAMERA, 3), dtype=np.uint8)
		cv2.putText(side_frame, "BRAK BOK", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

	front_frame = resize_frame(front_frame, MAX_WIDTH_PER_CAMERA)
	side_frame = resize_frame(side_frame, MAX_WIDTH_PER_CAMERA)

	target_h = int(max(front_frame.shape[0], side_frame.shape[0]))
	if front_frame.shape[0] != target_h:
		front_frame = cv2.resize(front_frame, (int(front_frame.shape[1]), int(target_h)))
	if side_frame.shape[0] != target_h:
		side_frame = cv2.resize(side_frame, (int(side_frame.shape[1]), int(target_h)))

	cv2.putText(front_frame, "FRONT", (16, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
	cv2.putText(side_frame, "BOK", (16, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

	# Nałóż FPS na klatki
	cv2.putText(front_frame, f"FPS: {get_fps('front')}", (16, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (180, 255, 180), 1)
	cv2.putText(side_frame, f"FPS: {get_fps('side')}", (16, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (180, 255, 180), 1)

	return np.hstack([front_frame, side_frame])


def render_status(active):
	dot = "#00ff88" if active else "#ff4444"
	label = "● NAGRYWANIE" if active else "● CZUWANIE"
	save_badge = "AUTO-ZAPIS ON" if save_cfg["enabled"] else "AUTO-ZAPIS OFF"
	status_html.value = f"""
	<div style='font-family:monospace;font-size:13px;color:{dot};
	background:#0d0d1a;padding:6px 14px;border-radius:8px;
	display:inline-block;letter-spacing:1px;margin-bottom:6px'>{label} | {save_badge}</div>"""


def render_voice(partial, final=""):
	voice_html.value = f"""
	<div style='font-family:monospace;font-size:12px;background:#0d0d1a;
	padding:6px 14px;border-radius:8px;margin-top:4px;border:1px solid #1e1e3a'>
		<span style='color:#555'>MIC:</span>
		<span style='color:#aaa'>{partial}</span>
		{"<span style='color:#00ff88;margin-left:8px'>-> " + final + "</span>" if final else ""}
	</div>"""


def render_assumptions():
	exercise = get_active_exercise()
	notes_html = "".join(f"<li>{note}</li>" for note in list(exercise["notes"]))
	commands_html = ", ".join(f"<code>{cmd}</code>" for cmd in VOICE_COMMANDS)
	assumptions_html.value = f"""
	<div style='font-family:monospace;background:#101322;border-radius:12px;padding:12px 18px;margin-bottom:6px;border:1px solid #1e1e3a'>
		<div style='color:#888;font-size:11px;letter-spacing:1px;margin-bottom:4px'>AKTYWNY SCENARIUSZ</div>
		<div style='color:#fff;font-size:16px;font-weight:bold;margin-bottom:6px'>{exercise['label']}</div>
		<div style='color:#bbb;font-size:12px;margin-bottom:6px'>{exercise['rep_rule']}</div>
		<ul style='color:#ddd;font-size:12px;margin:0 0 6px 18px;padding:0'>{notes_html}</ul>
		<div style='color:#888;font-size:11px'>Komendy: {commands_html}</div>
	</div>"""


def render_metrics():
	"""Widget z metrykami FPS i GT (ground truth) — aktualizowany co klatkę."""
	f_fps = get_fps("front")
	s_fps = get_fps("side")
	p_fps = get_fps("process")
	gt_block = ""
	if GT_MODE:
		with gt_lock:
			auto = dumbbell_state["reps"]
			manual = gt_manual_reps
		acc = ""
		if manual > 0:
			err = abs(auto - manual) / manual * 100
			acc = f"<span style='color:#ffcc00'>błąd: {err:.1f}%</span>"
		gt_block = f"""
		<div style='margin-top:6px;border-top:1px solid #1e1e3a;padding-top:6px'>
			<span style='color:#888;font-size:11px'>GROUND TRUTH</span><br>
			<span style='color:#aaa'>auto: {auto} | ręcznie: {manual} {acc}</span>
		</div>"""
	ram = get_ram_usage()
	gpu_mem, gpu_util = get_gpu_usage()
	gpu_str = f"GPU: {gpu_mem}MB {gpu_util}%" if gpu_mem else "GPU: brak"
	metrics_html.value = f"""
	<div style='font-family:monospace;background:#0a0a18;border-radius:8px;padding:8px 14px;margin-top:4px;border:1px solid #1e1e3a;font-size:12px'>
		<span style='color:#888'>FPS</span>
		<span style='color:#00ff88;margin-left:8px'>FRONT: {f_fps}</span>
		<span style='color:#ffcc00;margin-left:8px'>BOK: {s_fps}</span>
		<span style='color:#aaa;margin-left:8px'>PROC: {p_fps}</span>
		{gt_block}
		<span style='color:#ff88ff;margin-left:8px'>RAM: {ram}MB</span>
		<span style='color:#ff8844;margin-left:8px'>{gpu_str}</span>
	</div>"""


def render_stats(reps, avg_angle, good, reps_dict, source_label):
	exercise = get_active_exercise()

	recent_errors = []
	for r_key, r_val in reps_dict.items():
		rep_num = r_key.split("_")[-1]
		for err in r_val.get("errors", []):
			recent_errors.append({"rep": rep_num, "error": err["error"], "value": err["value"]})

	err_html = "".join([
		f"<div style='color:#ff6b6b;font-size:12px;margin-top:3px'>pow. {e['rep']}: {e['error']} ({e['value']})</div>"
		for e in recent_errors[-3:]
	])

	color = "#00ff88" if good else "#ff6b6b"
	phase_icon = dumbbell_state["phase"].upper()

	stats_html.value = f"""
	<div style='font-family:monospace;background:#0d0d1a;border-radius:12px;padding:12px 18px;margin-top:6px;border:1px solid #1e1e3a'>
		<div style='display:flex;gap:24px;align-items:center;flex-wrap:wrap'>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>CWICZENIE</div>
				<div style='color:#fff;font-size:18px;font-weight:bold'>{exercise['label']}</div>
			</div>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>POWTORZENIA</div>
				<div style='color:#fff;font-size:28px;font-weight:bold'>{reps}</div>
			</div>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>KAT LOKCIA</div>
				<div style='color:{color};font-size:28px;font-weight:bold'>{avg_angle:.0f}</div>
			</div>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>FAZA</div>
				<div style='color:#aaa;font-size:20px'>{phase_icon}</div>
			</div>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>ANALIZA</div>
				<div style='color:#aaa;font-size:14px'>{source_label}</div>
			</div>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>FORMA</div>
				<div style='color:{color};font-size:16px;font-weight:bold'>{'OK' if good else 'DOKONCZ RUCH'}</div>
			</div>
		</div>
		{err_html}
	</div>"""


def gt_increment():
	"""Ręczne zliczenie powtórzenia w trybie GT — wywołaj z zewnątrz lub klawiszem."""
	global gt_manual_reps
	with gt_lock:
		gt_manual_reps += 1
	print(f"[GT] Ręczne powtórzenie: {gt_manual_reps}")


def gt_reset():
	global gt_manual_reps
	with gt_lock:
		gt_manual_reps = 0


def reset_motion_state(reset_reps=True):
	dumbbell_state["phase"] = "extended"
	dumbbell_state["cycle_min"] = 180.0
	dumbbell_state["cycle_max"] = 0.0
	dumbbell_state["amplitude_warned"] = False
	dumbbell_state["symmetry_warned"] = False
	dumbbell_state["sway_warned"] = False
	dumbbell_state["elbow_warned"] = False
	dumbbell_state["candidate_phase"] = None
	dumbbell_state["candidate_count"] = 0
	dumbbell_state["last_rep_time"] = 0
	if reset_reps:
		dumbbell_state["reps"] = 0
	if GT_MODE:
		gt_reset()


def analyze_dumbbell_curl(front_landmarks, side_landmarks):
	exercise = get_active_exercise()
	front_avg = None
	side_avg = None
	fl = fr = sl = sr = None
	try:
		if front_landmarks is not None:
			fl, fr, front_avg = arm_angles(front_landmarks)
			_log_confidence("front", front_landmarks)
		if side_landmarks is not None:
			sl, sr, side_avg = arm_angles(side_landmarks)
			_log_confidence("side", side_landmarks)
	except Exception:
		pass

	if front_avg is None and side_avg is None:
		return dumbbell_state["reps"], 0.0, True, "brak sylwetki"

	if front_avg is not None and side_avg is not None:
		raw_angle = (front_avg + side_avg) / 2
		angle_buffer.append(raw_angle)
		avg_angle = sum(angle_buffer) / len(angle_buffer)
		both_available = True
		source = "both"
	else:
		raw_angle = side_avg if side_avg is not None else front_avg
		angle_buffer.append(raw_angle)
		avg_angle = sum(angle_buffer) / len(angle_buffer)
		both_available = False
		source = "side" if side_avg is not None else "front"

	# Log kąta do CSV
	_log_angle(
		left=fl if fl is not None else sl,
		right=fr if fr is not None else sr,
		avg=raw_angle,
		smoothed=avg_angle,
		phase=dumbbell_state["phase"],
		source=source,
	)

	if BOTH_CAMERAS_REQUIRED and not both_available:
		return dumbbell_state["reps"], float(avg_angle), True, "oczekuje na obie kamery"

	dumbbell_state["cycle_min"] = min(dumbbell_state.get("cycle_min", 180.0), float(avg_angle))
	dumbbell_state["cycle_max"] = max(dumbbell_state.get("cycle_max", 0.0), float(avg_angle))

	down_thresh = exercise["down_threshold"]
	up_thresh = exercise["up_threshold"]

	if avg_angle < (down_thresh - HYSTERESIS):
		instant_phase = "flexed"
	elif avg_angle > (up_thresh + HYSTERESIS):
		instant_phase = "extended"
	else:
		instant_phase = dumbbell_state["phase"]

	if instant_phase != dumbbell_state["phase"]:
		if dumbbell_state["candidate_phase"] == instant_phase:
			dumbbell_state["candidate_count"] += 1
		else:
			dumbbell_state["candidate_phase"] = instant_phase
			dumbbell_state["candidate_count"] = 1

		if dumbbell_state["candidate_count"] >= STATE_CONFIRM_FRAMES:
			old_phase = dumbbell_state["phase"]
			dumbbell_state["phase"] = instant_phase
			dumbbell_state["candidate_phase"] = None
			dumbbell_state["candidate_count"] = 0

			if old_phase == "flexed" and instant_phase == "extended":
				now = time.time()
				if recording["active"] and now - dumbbell_state["last_rep_time"] >= MIN_REP_INTERVAL:
					dumbbell_state["reps"] += 1
					dumbbell_state["last_rep_time"] = now

					current_rep = dumbbell_state["reps"]
					rep_key = f"rep_{current_rep}"
					amplitude = dumbbell_state["cycle_max"] - dumbbell_state["cycle_min"]

					if rep_key not in recording["stats_by_rep"]:
						recording["stats_by_rep"][rep_key] = {"errors": []}

					recording["stats_by_rep"][rep_key].update({
						"info": "udane powtorzenie",
						"min_angle": round(dumbbell_state["cycle_min"], 1),
						"max_angle": round(dumbbell_state["cycle_max"], 1),
						"amplitude": round(amplitude, 1)
					})

					if amplitude < exercise["min_amplitude"]:
						recording["stats_by_rep"][rep_key]["errors"].append({
							"error": "Niewystarczajaca amplituda",
							"value": round(amplitude, 1),
							"camera": "both" if both_available else "side"
						})
						speak("wyzej")

					speak(f"powtorzenie {current_rep}")

					dumbbell_state["cycle_min"] = avg_angle
					dumbbell_state["cycle_max"] = avg_angle
	else:
		dumbbell_state["candidate_phase"] = None
		dumbbell_state["candidate_count"] = 0

	current_rep = dumbbell_state["reps"] + 1
	rep_key = f"rep_{current_rep}"

	if recording["active"]:
		if rep_key not in recording["stats_by_rep"]:
			recording["stats_by_rep"][rep_key] = {"errors": []}

		if front_avg is not None:
			try:
				fl2, fr2, _ = arm_angles(front_landmarks)
				gap = abs(fl2 - fr2)
				if gap > exercise["symmetry_threshold"] and not dumbbell_state.get("symmetry_warned", False):
					recording["stats_by_rep"][rep_key]["errors"].append({
						"error": "Asymetria ramion",
						"value": round(gap, 1),
						"camera": "front"
					})
					dumbbell_state["symmetry_warned"] = True
				elif gap <= exercise["symmetry_threshold"]:
					dumbbell_state["symmetry_warned"] = False
			except Exception:
				pass

		if side_landmarks is not None:
			try:
				shoulder = side_landmarks[11]
				hip = side_landmarks[23]
				dx = shoulder.x - hip.x
				dy = shoulder.y - hip.y
				sway_angle = abs(np.degrees(np.arctan2(dx, -dy)))

				if sway_angle > exercise.get("sway_threshold", 15) and not dumbbell_state.get("sway_warned", False):
					recording["stats_by_rep"][rep_key]["errors"].append({
						"error": "Bujanie tulowiem",
						"value": round(sway_angle, 1),
						"camera": "side"
					})
					speak("nie bujaj")
					dumbbell_state["sway_warned"] = True
				elif sway_angle <= exercise.get("sway_threshold", 15):
					dumbbell_state["sway_warned"] = False
			except Exception:
				pass

		if side_landmarks is not None:
			try:
				shoulder = side_landmarks[11]
				elbow = side_landmarks[13]
				hip = side_landmarks[23]
				sway_angle = abs(np.degrees(np.arctan2(shoulder.x - hip.x, -(shoulder.y - hip.y))))

				arm_dx = elbow.x - shoulder.x
				arm_dy = elbow.y - shoulder.y
				arm_angle_vert = abs(np.degrees(np.arctan2(arm_dx, -arm_dy))) - sway_angle

				if abs(arm_angle_vert) > exercise.get("elbow_threshold", 25) and not dumbbell_state.get("elbow_warned", False):
					recording["stats_by_rep"][rep_key]["errors"].append({
						"error": "Uciekajacy lokiec",
						"value": round(abs(arm_angle_vert), 1),
						"camera": "side"
					})
					speak("lokcie blisko")
					dumbbell_state["elbow_warned"] = True
				elif abs(arm_angle_vert) <= exercise.get("elbow_threshold", 25):
					dumbbell_state["elbow_warned"] = False
			except Exception:
				pass

	good = (avg_angle < (down_thresh - HYSTERESIS) + 10) or (avg_angle > (up_thresh + HYSTERESIS) - 10)
	return dumbbell_state["reps"], float(avg_angle), good, source


def save_session(reason="stop"):
	recording["active"] = False
	with state_lock:
		proc = recording.get("video_writer")
		if proc is not None:
			try:
				proc.stdin.close()
				proc.wait(timeout=10)
			except Exception:
				proc.kill()
			recording["video_writer"] = None
		recording["_video_size"] = None

	# Flush buforów logów
	with _angle_log_lock:
		_flush_angle_log()

	ts = datetime.now().strftime("%Y%m%d_%H%M%S")
	series_idx = recording.get("series_index", 1)
	Path("sessions").mkdir(exist_ok=True)
	stats_path = f"sessions/stats_{ts}_series_{series_idx}.json"
	video_path = recording.get("video_path")

	# Ground truth do JSON
	gt_data = {}
	if GT_MODE:
		with gt_lock:
			auto = dumbbell_state.get("reps", 0)
			manual = gt_manual_reps
		gt_data = {
			"gt_manual_reps": manual,
			"gt_auto_reps": auto,
			"gt_error_pct": round(abs(auto - manual) / manual * 100, 2) if manual > 0 else None,
		}

	# FPS snapshot do JSON
	fps_snapshot = {
		"fps_front": get_fps("front"),
		"fps_side": get_fps("side"),
		"fps_process": get_fps("process"),
	}

	session_payload = {
		"exercise": ACTIVE_EXERCISE_KEY,
		"exercise_label": get_active_exercise()["label"],
		"reps": dumbbell_state.get("reps", 0),
		"reps_detail": recording.get("stats_by_rep", {}),
		"series_index": series_idx,
		"reason": reason,
		"auto_save_enabled": save_cfg.get("enabled", True),
		"video_path": video_path,
		"voice_commands": VOICE_COMMANDS,
		"started_at": recording.get("started_at"),
		"angle_log_path": _angle_log_path,
		"fps": fps_snapshot,
		"ram_mb": get_ram_usage(),
		"gpu_mem_mb": get_gpu_usage()[0],
		"gpu_util_pct": get_gpu_usage()[1],
		**gt_data,
	}

	_flush_confidence_log(f"sessions/conf_{ts}_series_{series_idx}")

	with open(stats_path, "w", encoding="utf-8") as file:
		json.dump(session_payload, file, indent=2, ensure_ascii=False)

	render_voice("", f"zapisano serie {series_idx}: {dumbbell_state.get('reps', 0)} powtorzen")
	recording["series_index"] = series_idx + 1
	recording["frames"] = []
	recording["video_path"] = None
	recording["stats_by_rep"] = {}
	recording["started_at"] = None


def begin_series(reset_reps=True):
	reset_motion_state(reset_reps=reset_reps)
	recording["active"] = True
	recording["frames"] = []
	recording["stats_by_rep"] = {}
	recording["started_at"] = time.time()
	_init_logs()
	render_status(True)


def end_series(reason="manual"):
	if not recording["active"]:
		return
	save_session(reason=reason)
	render_status(False)

def draw_pose_roi(frame, landmarks, bbox, line_color=(0, 255, 136), point_color=(255, 100, 100)):
	x1, y1, x2, y2 = bbox
	rw, rh = x2 - x1, y2 - y1
	for start_idx, end_idx in POSE_CONNECTIONS:
		p1 = landmarks[start_idx]
		p2 = landmarks[end_idx]
		px1 = int(p1.x * rw) + x1
		py1 = int(p1.y * rh) + y1
		px2 = int(p2.x * rw) + x1
		py2 = int(p2.y * rh) + y1
		cv2.line(frame, (px1, py1), (px2, py2), line_color, 2)
	for lm in landmarks:
		cx = int(lm.x * rw) + x1
		cy = int(lm.y * rh) + y1
		cv2.circle(frame, (cx, cy), 4, point_color, -1)

YOLO_EVERY_N_FRAMES = 10
_yolo_bbox_cache = {"front": None, "side": None}
_yolo_frame_counter = {"front": 0, "side": 0}

def fetch_camera(camera_name, stream_url, mirror=True):
	cap = cv2.VideoCapture(stream_url)
	cap.set(cv2.CAP_PROP_FPS, FPS)

	thread_options = PoseLandmarkerOptions(
		base_options=BaseOptions(model_asset_path=MODEL_PATH),
		running_mode=VisionRunningMode.VIDEO,
		num_poses=1,
	)

	last_timestamp = 0
	with PoseLandmarker.create_from_options(thread_options) as landmarker:
		while cap.isOpened() and not stop.is_set():
			ret, frame = cap.read()
			if not ret:
				cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
				continue

			if mirror:
				frame = cv2.flip(frame, 1)

			with _yolo_lock:
				bbox = None

			if bbox is not None:
				x1, y1, x2, y2 = bbox
				cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 200, 255), 2)
				roi = frame[y1:y2, x1:x2]
			else:
				roi = frame

			rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
			mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

			current_timestamp = int(time.time() * 1000)
			if current_timestamp <= last_timestamp:
				current_timestamp = last_timestamp + 1
			last_timestamp = current_timestamp

			fps_counters[camera_name].append(time.time())

			try:
				result = landmarker.detect_for_video(mp_image, current_timestamp)
				landmarks = result.pose_landmarks[0] if result.pose_landmarks else None
			except Exception as e:
				print(f"[MediaPipe Error ({camera_name})]: {e}")
				landmarks = None

			if landmarks is not None:
				color = (0, 255, 136) if camera_name == "front" else (255, 196, 0)
				if bbox is not None:
					draw_pose_roi(frame, landmarks, bbox, line_color=color)
				else:
					draw_pose(frame, landmarks, line_color=color)

			with state_lock:
				latest_frames[camera_name] = frame
				latest_landmarks[camera_name] = landmarks
				latest_timestamps[camera_name] = current_timestamp
	cap.release()


def process_loop():
	while not stop.is_set():
		with state_lock:
			f_ts = latest_timestamps["front"]
			s_ts = latest_timestamps["side"]

			if f_ts == processed_timestamps["front"] and s_ts == processed_timestamps["side"]:
				has_new_data = False
			else:
				has_new_data = True
				processed_timestamps["front"] = f_ts
				processed_timestamps["side"] = s_ts
				front_frame = None if latest_frames["front"] is None else latest_frames["front"].copy()
				side_frame = None if latest_frames["side"] is None else latest_frames["side"].copy()
				front_landmarks = latest_landmarks["front"]
				side_landmarks = latest_landmarks["side"]

		if not has_new_data:
			time.sleep(0.005)
			continue

		fps_counters["process"].append(time.time())

		reps, avg_angle, good, source = analyze_dumbbell_curl(front_landmarks, side_landmarks)
		render_stats(reps, avg_angle, good, recording["stats_by_rep"], source)
		render_metrics()

		composed = compose_frames(front_frame, side_frame)
		if composed is None:
			time.sleep(0.03)
			continue

		if recording["active"] and save_cfg.get("enabled", True):
			if recording.get("video_writer") is None:
				h, w = composed.shape[:2]
				w = (w // 2) * 2
				h = (h // 2) * 2
				composed = cv2.resize(composed, (w, h))
				ts = datetime.now().strftime("%Y%m%d_%H%M%S")
				series_idx = recording.get("series_index", 1)
				Path("sessions").mkdir(exist_ok=True)
				vp = f"sessions/session_{ts}_series_{series_idx}.mp4"
				recording["video_path"] = vp
				recording["_video_size"] = (w, h)
				recording["video_writer"] = _create_ffmpeg_writer(vp, w, h, FPS)
			try:
				target_size = recording.get("_video_size")
				if target_size:
					tw, th = target_size
					if composed.shape[1] != tw or composed.shape[0] != th:
						composed = cv2.resize(composed, (tw, th))
				recording["video_writer"].stdin.write(composed.tobytes())
			except Exception:
				pass

		ok, buffer = cv2.imencode(".jpg", composed, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
		if ok:
			img_widget.value = buffer.tobytes()

		time.sleep(0.01)


def _apply_save_toggle(cmd):
	normalized = cmd.strip().lower()
	normalized = re.sub(r"\s+", " ", normalized)
	if "zapis off" in normalized:
		save_cfg["enabled"] = False
		render_status(recording["active"])
		render_voice("", "auto-zapis wylaczony")
		speak("auto zapis off")
		return True
	if "zapis on" in normalized:
		save_cfg["enabled"] = True
		render_status(recording["active"])
		render_voice("", "auto-zapis wlaczony")
		speak("auto zapis on")
		return True
	return False


def stop_everything(reason="stop"):
	if stop.is_set():
		return
	end_series(reason=reason)
	stop.set()


def listen_loop():
	print("[VOICE] Dostępne urządzenia audio:")
	print(sd.query_devices())
	print(f"[VOICE] Domyślne wejście: {sd.query_devices(kind='input')['name']}")
	all_allowed_words = START_WORDS + STOP_WORDS + SERIES_WORDS + [
		"reset", "zeruj", "zapis on", "zapis off", "[unk]"
	]
	grammar_json = json.dumps(all_allowed_words, ensure_ascii=False)

	try:
		rec = vosk.KaldiRecognizer(vosk_model, 16000, grammar_json)
	except Exception as e:
		print(f"[Vosk Warning] Tryb pelny: {e}")
		rec = vosk.KaldiRecognizer(vosk_model, 16000)

	mic_device = None

	while not stop.is_set():
		try:
			with sd.RawInputStream(samplerate=16000, channels=1, dtype="int16", device=mic_device) as stream:
				print("[VOICE] Mikrofon uruchomiony.")
				while not stop.is_set():
					if speech_lock.locked():
						time.sleep(0.1)
						continue

					data, _ = stream.read(4000)
					if len(data) == 0:
						continue

					if rec.AcceptWaveform(bytes(data)):
						res = json.loads(rec.Result())
						cmd = res.get("text", "").lower().strip()
						cmd = re.sub(r"[^\w\s]", "", cmd)
						print(f"[VOSK RAW] '{cmd}'")
						if not cmd or cmd == "[unk]":
							continue

						render_voice("", cmd)
						if _apply_save_toggle(cmd):
							continue

						normalized = re.sub(r"\s+", " ", cmd)
						if any(w in normalized for w in START_WORDS):
							begin_series(reset_reps=True)
							speak("start")
						elif any(w in normalized for w in SERIES_WORDS):
							end_series(reason="next_series")
							begin_series(reset_reps=True)
							speak("nowa seria")
						elif "reset" in normalized or "zeruj" in normalized:
							reset_motion_state(reset_reps=True)
							speak("reset")
						elif any(w in normalized for w in STOP_WORDS):
							speak(f"koniec powtorzenia {dumbbell_state.get('reps', 0)}")
							stop_everything(reason="voice_stop")
					else:
						partial = json.loads(rec.PartialResult()).get("partial", "")
						if partial and partial != "[unk]":
							render_voice(partial)

		except Exception as e:
			print(f"[Audio Glitch] {e}")
			time.sleep(0.4)


def start():
	if not Path(MODEL_PATH).exists() or not Path(VOSK_MODEL_PATH).exists():
		print("BLAD: Brak modelu MediaPipe lub Vosk na dysku. Upewnij sie, ze pliki sa we wlasciwym miejscu.")
		return

	stop.clear()
	render_assumptions()
	render_voice("gotowy...")

	reset_motion_state(reset_reps=True)
	render_stats(0, 0, True, {}, "oczekiwanie")
	render_metrics()

	if AUTO_START_SERIES:
		begin_series(reset_reps=True)
	else:
		render_status(False)

	display(widgets.VBox([
		assumptions_html,
		status_html,
		img_widget,
		stats_html,
		metrics_html,
		voice_html,
	], layout=widgets.Layout(gap="4px")))

	threading.Thread(target=tts_worker, daemon=True).start()
	threading.Thread(target=fetch_camera, args=("front", FRONT_STREAM_URL, True), daemon=True).start()
	threading.Thread(target=fetch_camera, args=("side", SIDE_STREAM_URL, True), daemon=True).start()
	threading.Thread(target=process_loop, daemon=True).start()
	threading.Thread(target=listen_loop, daemon=True).start()
	threading.Thread(target=_gpu_worker, daemon=True).start()

	if TIMEOUT > 0:
		threading.Timer(TIMEOUT, lambda: stop_everything(reason="timeout")).start()


def playback_session(stats_path, video_path):
	with open(stats_path, encoding="utf-8") as file:
		stats = json.load(file)

	# Pobieramy nową, ustrukturyzowaną formę błędów pogrupowaną po powtórzeniach
	reps_data = stats.get("reps_detail", {})
	cap = cv2.VideoCapture(video_path)
	total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
	cap.release()

	if not total_frames:
		print("Brak klatek w nagraniu.")
		return

	playback_widget = widgets.Image(format="jpeg", layout=widgets.Layout(width="100%"))
	info_html = widgets.HTML()

	def show(idx):
		cap = cv2.VideoCapture(video_path)
		cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
		ret, frame = cap.read()
		cap.release()
		if not ret:
			return

		frame = frame.copy()
		rep_num = max(1, round(idx / max(1, total_frames) * max(1, stats.get("reps", 0))))
		rep_key = f"rep_{rep_num}"

		# Jeśli to powtórzenie miało zarejestrowane jakieś błędy, rysujemy czerwoną ramkę
		if rep_key in reps_data and reps_data[rep_key].get("errors"):
			cv2.rectangle(frame, (0, 0), (frame.shape[1], frame.shape[0]), (0, 0, 255), 6)
			errs = reps_data[rep_key]["errors"]
			cv2.putText(frame, f"BLAD: {errs[0]['error']} ({errs[0]['value']})", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

		ok, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 75])
		if ok:
			playback_widget.value = buf.tobytes()

		total_errors = sum(len(v.get("errors", [])) for v in reps_data.values())
		info_html.value = (
			f"<div style='font-family:monospace;color:#aaa'>"
			f"klatka {idx + 1}/{total_frames} | powtorzenia: {stats.get('reps', 0)} | "
			f"lacznie bledow: {total_errors} | seria: {stats.get('series_index', '?')}"
			f"</div>"
		)

	slider = widgets.IntSlider(min=0, max=total_frames - 1, step=1, layout=widgets.Layout(width="100%"))
	widgets.interact(show, idx=slider)
	display(widgets.VBox([playback_widget, info_html]))


def playback_latest():
	sessions = sorted(Path("sessions").glob("stats_*_series_*.json"))
	if not sessions:
		print("Brak sesji.")
		return

	stats_path = str(sessions[-1])
	with open(stats_path, encoding="utf-8") as f:
		stats = json.load(f)
	video_path = stats.get("video_path")
	if not video_path or not Path(video_path).exists():
		print("Brak pliku wideo dla tej sesji.")
		return
	playback_session(stats_path, video_path)

In [ ]:
start()


[VOICE] Dostępne urządzenia audio:
   0 HDA Intel PCH: ALC897 Analog (hw:0,0), ALSA (2 in, 0 out)
   1 HDA Intel PCH: ALC897 Digital (hw:0,1), ALSA (0 in, 2 out)
   2 HDA Intel PCH: ALC897 Alt Analog (hw:0,2), ALSA (2 in, 0 out)
   3 HDA ATI HDMI: 24M2N3200 (hw:1,3), ALSA (0 in, 2 out)
   4 HDA ATI HDMI: 24M2N3200 (hw:1,7), ALSA (0 in, 2 out)
   5 HDA ATI HDMI: 2 (hw:1,8), ALSA (0 in, 8 out)
   6 HDA ATI HDMI: 3 (hw:1,9), ALSA (0 in, 8 out)
   7 sysdefault, ALSA (128 in, 0 out)
   8 iec958, ALSA (0 in, 2 out)
   9 spdif, ALSA (0 in, 2 out)
  10 lavrate, ALSA (128 in, 0 out)
  11 samplerate, ALSA (128 in, 0 out)
  12 speexrate, ALSA (128 in, 0 out)
  13 pipewire, ALSA (128 in, 128 out)
  14 pulse, ALSA (32 in, 32 out)
  15 a52, ALSA (0 in, 6 out)
  16 speex, ALSA (1 in, 0 out)
  17 upmix, ALSA (8 in, 0 out)
  18 vdownmix, ALSA (6 in, 0 out)
* 19 default, ALSA (128 in, 128 out)
[VOICE] Domyślne wejście: default


WARNING (VoskAPI:UpdateGrammarFst():recognizer.cc:308) Ignoring word missing in vocabulary: 'skoncz'
WARNING (VoskAPI:UpdateGrammarFst():recognizer.cc:308) Ignoring word missing in vocabulary: 'koncz'
WARNING (VoskAPI:UpdateGrammarFst():recognizer.cc:308) Ignoring word missing in vocabulary: 'zeruj'


[VOICE] Mikrofon uruchomiony.


I0000 00:00:1781983700.171676  125619 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1781983700.173625  125640 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.1), renderer: AMD Radeon RX 7900 XT (radeonsi, navi31, LLVM 20.1.2, DRM 3.64, 6.18.7-76061807-generic)
I0000 00:00:1781983700.182079  125657 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1781983700.183789  125678 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.1), renderer: AMD Radeon RX 7900 XT (radeonsi, navi31, LLVM 20.1.2, DRM 3.64, 6.18.7-76061807-generic)
W0000 00:00:1781983700.226147  125633 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781983700.230878  125671 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for 

[VOSK RAW] 'start'
[VOSK RAW] 'stop'


In [31]:
stop.set()
#playback_latest()


In [53]:
import csv
import json
import sys
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

SESSION_DIR = Path("sessions")
OUT_DIR = Path("plots")
OUT_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
	"figure.facecolor": "#0d0d1a",
	"axes.facecolor": "#0d0d1a",
	"axes.edgecolor": "#333355",
	"axes.labelcolor": "#cccccc",
	"xtick.color": "#cccccc",
	"ytick.color": "#cccccc",
	"text.color": "#cccccc",
	"grid.color": "#1e1e3a",
	"grid.linestyle": "--",
})


def load_latest_angle_csv():
	files = sorted(SESSION_DIR.glob("angles_*.csv"))
	if not files:
		print("Brak pliku angles_*.csv w sessions/")
		sys.exit(1)
	return files[-1]


def load_latest_stats_json():
	files = sorted(SESSION_DIR.glob("stats_*.json"))
	if not files:
		print("Brak pliku stats_*.json w sessions/")
		sys.exit(1)
	return files[-1]


def plot_angle_over_time(csv_path):
	timestamps, smoothed, phases = [], [], []
	with open(csv_path, newline="", encoding="utf-8") as f:
		for row in csv.DictReader(f):
			try:
				timestamps.append(float(row["timestamp"]))
				smoothed.append(float(row["smoothed_angle"]) if row["smoothed_angle"] else None)
				phases.append(row["phase"])
			except ValueError:
				continue

	if not timestamps:
		print("Pusty plik CSV")
		return

	t0 = timestamps[0]
	t = [x - t0 for x in timestamps]
	s = [x if x is not None else float("nan") for x in smoothed]

	fig, ax = plt.subplots(figsize=(12, 4))
	ax.plot(t, s, color="#00ff88", linewidth=1.2, label="Kąt wygładzony [°]")

	for i in range(len(phases) - 1):
		if phases[i] == "flexed":
			ax.axvspan(t[i], t[i + 1], alpha=0.15, color="#ff6b6b")

	ax.axhline(40, color="#ff4444", linestyle="--", linewidth=0.8, label="Próg ugięcia (40°)")
	ax.axhline(148, color="#4444ff", linestyle="--", linewidth=0.8, label="Próg wyprostu (148°)")
	ax.set_xlabel("Czas [s]")
	ax.set_ylabel("Kąt łokcia [°]")
	ax.set_title("Rysunek 1. Kąt łokcia w funkcji czasu (czerwone tło = faza ugięcia)")
	ax.legend(loc="upper right", fontsize=8)
	ax.grid(True)
	fig.tight_layout()
	out = OUT_DIR / "fig1_angle_over_time.png"
	fig.savefig(out, dpi=150)
	plt.close(fig)
	print(f"Zapisano: {out}")


def plot_amplitude_per_rep(json_path):
	with open(json_path, encoding="utf-8") as f:
		data = json.load(f)

	reps_detail = data.get("reps_detail", {})
	rep_nums, amplitudes, errors = [], [], []

	for key, val in reps_detail.items():
		if "amplitude" not in val:
			continue
		rep_nums.append(int(key.split("_")[-1]))
		amplitudes.append(val["amplitude"])
		errors.append(len(val.get("errors", [])))

	if not rep_nums:
		print("Brak danych amplitudy w JSON")
		return

	colors = ["#ff6b6b" if e > 0 else "#00ff88" for e in errors]
	fig, ax = plt.subplots(figsize=(max(6, len(rep_nums) * 0.8), 4))
	bars = ax.bar(rep_nums, amplitudes, color=colors, edgecolor="#0d0d1a", width=0.6)
	ax.axhline(100, color="#ffcc00", linestyle="--", linewidth=0.8, label="Min. amplituda (100°)")
	ax.set_xlabel("Numer powtórzenia")
	ax.set_ylabel("Amplituda ruchu [°]")
	ax.set_title("Rysunek 2. Amplituda ruchu na powtórzenie (czerwony = błąd formy)")
	ax.set_xticks(rep_nums)
	ax.legend(fontsize=8)
	ax.grid(True, axis="y")
	fig.tight_layout()
	out = OUT_DIR / "fig2_amplitude_per_rep.png"
	fig.savefig(out, dpi=150)
	plt.close(fig)
	print(f"Zapisano: {out}")


def plot_gt_table(json_path):
	with open(json_path, encoding="utf-8") as f:
		data = json.load(f)

	if "gt_manual_reps" not in data:
		print("Brak danych GT w JSON (GT_MODE był False)")
		return

	rows = [
		["Seria", "Auto", "Ręcznie", "Błąd [%]"],
		[
			str(data.get("series_index", 1)),
			str(data.get("gt_auto_reps", "?")),
			str(data.get("gt_manual_reps", "?")),
			str(data.get("gt_error_pct", "?")) + "%" if data.get("gt_error_pct") is not None else "N/A",
		],
	]

	fig, ax = plt.subplots(figsize=(5, 1.5))
	ax.axis("off")
	tbl = ax.table(cellText=rows[1:], colLabels=rows[0], loc="center", cellLoc="center")
	tbl.auto_set_font_size(False)
	tbl.set_fontsize(11)
	tbl.scale(1, 2)
	for (r, c), cell in tbl.get_celld().items():
		cell.set_edgecolor("#333355")
		cell.set_facecolor("#1a1a2e" if r > 0 else "#0a0a30")
		cell.set_text_props(color="#cccccc")
	ax.set_title("Tabela 1. Porównanie liczenia automatycznego z ręcznym (Ground Truth)", pad=12, fontsize=9)
	fig.tight_layout()
	out = OUT_DIR / "tab1_ground_truth.png"
	fig.savefig(out, dpi=150, bbox_inches="tight")
	plt.close(fig)
	print(f"Zapisano: {out}")


def plot_fps_history(csv_path):
	timestamps = []
	with open(csv_path, newline="", encoding="utf-8") as f:
		for row in csv.DictReader(f):
			try:
				timestamps.append(float(row["timestamp"]))
			except ValueError:
				continue

	if len(timestamps) < 10:
		print("Za mało danych do wykresu FPS")
		return

	window = 30
	t0 = timestamps[0]
	fps_vals, t_vals = [], []
	for i in range(window, len(timestamps)):
		dt = timestamps[i] - timestamps[i - window]
		if dt > 0:
			fps_vals.append(window / dt)
			t_vals.append(timestamps[i] - t0)

	fig, ax = plt.subplots(figsize=(12, 3))
	ax.plot(t_vals, fps_vals, color="#ffcc00", linewidth=1.0)
	ax.axhline(np.mean(fps_vals), color="#00ff88", linestyle="--", linewidth=0.8,
		label=f"Średnie FPS: {np.mean(fps_vals):.1f}")
	ax.set_xlabel("Czas [s]")
	ax.set_ylabel("FPS")
	ax.set_title("Rysunek 3. Liczba klatek na sekundę przetwarzanej analizy w czasie")
	ax.legend(fontsize=8)
	ax.grid(True)
	fig.tight_layout()
	out = OUT_DIR / "fig3_fps_history.png"
	fig.savefig(out, dpi=150)
	plt.close(fig)
	print(f"Zapisano: {out}")


if __name__ == "__main__":
	angle_csv = load_latest_angle_csv()
	stats_json = load_latest_stats_json()
	print(f"Używam: {angle_csv.name}, {stats_json.name}")
	plot_angle_over_time(angle_csv)
	plot_amplitude_per_rep(stats_json)
	plot_gt_table(stats_json)
	plot_fps_history(angle_csv)
	print("Gotowe. Pliki w katalogu plots/")

Używam: angles_20260620_212823.csv, stats_20260620_212839_series_1.json
Zapisano: plots/fig1_angle_over_time.png
Zapisano: plots/fig2_amplitude_per_rep.png
Zapisano: plots/tab1_ground_truth.png
Zapisano: plots/fig3_fps_history.png
Gotowe. Pliki w katalogu plots/
